# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Razlaganje zadatka

Razlaganje zadatka je srž obrasca dizajna planiranja. Umjesto da od jednog agenta tražimo
da riješi složen zahtjev od početka do kraja, problem razbijamo u manje, dobro definirane **podzadatke**.
Svaki podzadatak dodjeljuje se specijaliziranom agentu (npr. letovi, hoteli, aktivnosti) s jasnim
prioritetima i redoslijedom ovisnosti.

Ovaj pristup donosi nekoliko pogodnosti:
- **Jasnoća**: svaki podzadatak ima jednu odgovornost.
- **Paralelizam**: neovisni podzadatci mogu se izvršavati istovremeno.
- **Pouzdanost**: kvarovi su izolirani na pojedinačne podzadatke.
- **Praćenje budžeta**: troškovi se procjenjuju po podzadatku i zbrajaju.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Izrada planerskog agenta sa strukturiranim izlazom

Planerski agent djeluje kao **koordinator na recepciji**. S obzirom na zahtjev za putovanje na visokoj razini,
proizvodi strukturirani `TravelPlan` — razlažući zahtjev na podzadatke, postavljajući prioritete,
i identificirajući ovisnosti kako bi osoba na recepciji ili sloj za izvršenje mogli obaviti posao.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Izvršavanje plana s pomoćnim alatima

Nakon što je agent na recepciji izradio strukturirani plan, **agent konsijerž** ga izvršava.
Svaki specijalizirani alat upravlja jednom kategorijom podzadatka (leti, hoteli, aktivnosti). Konsijerž
prolazi kroz podzadatke u planu redom ovisnosti i upućuje svaki odgovarajućem
alatu.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Sažetak

U ovoj lekciji naučili ste **Predložak dizajna planiranja** za AI agente:

1. **Razlaganje zadatka** — Agent za planiranje na recepciji razlaže složen zahtjev za putovanje u
   strukturirane podzadatke koristeći Pydantic modele, dodjeljujući svaki specijaliziranom agentu s prioritetima
   i ovisnostima.
2. **Strukturirani izlaz** — Prosljeđivanjem `response_format` agent vraća potvrđeni
   objekt `TravelPlan` umjesto slobodnog teksta, što čini daljnju obradu pouzdanom.
3. **Izvršenje plana** — Concierge agent iterira kroz podzadatke koristeći specijalizirane alate
   (`book_flight`, `reserve_hotel`, `book_activity`) za provođenje plana i izvještavanje o rezultatima.

Ovaj predložak odvaja *što treba napraviti* (planiranje) od *kako to napraviti* (izvršenje), čineći agente
modularnijima, testabilnijima i lakšima za proširenje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
